In [ ]:
!pip install chardet
# !pip uninstall -y tensorflow && pip install tensorflow-cpu

In [ ]:
import pandas as pd
import numpy as np
import chardet
import re
import requests
import time
import os
import scipy.sparse as sp
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Load JSONL file into a DataFrame
data_ru = pd.read_json('/content/drive/MyDrive/Colab Notebooks/train.jsonl', lines=True)

In [ ]:
# Load the Helsinki-NLP translation model
model_name = "Helsinki-NLP/opus-mt-ru-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

In [ ]:
def split_and_translate(text, max_tokens=512):
    # Ensure text is a string and tokenize sentences
    sentences = text.split(". ")  # Split into sentences
    chunks, current_chunk = [], ""

    # Divide text into manageable chunks
    for sentence in sentences:
        if len(tokenizer.encode(current_chunk + sentence, truncation=False)) < max_tokens:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:  # Add the last chunk
        chunks.append(current_chunk.strip())

    # Translate each chunk and combine results
    translated_chunks = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=max_tokens)
        outputs = model.generate(**inputs)
        translated_chunks.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

    return " ".join(translated_chunks)


In [ ]:
# Define the output file path for the translated data
output_path = '/content/drive/MyDrive/translated_data.csv'

# Load the existing translated data if it exists
try:
    # Check if the output file already exists and read it
    existing_data = pd.read_csv(output_path, encoding='utf-8')
    start_index = len(existing_data)  # Set the start index to continue from the last processed row
    print(f"Existing data with {start_index} rows found. Continuing processing.")
except FileNotFoundError:
    # If the file does not exist, start fresh
    existing_data = pd.DataFrame(columns=['dialog_translated', 'summary_translated'])
    start_index = 0
    print("No existing file found. Starting new translation process.")

# Iterate through the dataset in batches
batch_size = 10  # Define the batch size for processing
for i in range(start_index, len(data_ru), batch_size):
    try:
        batch_data = []  # List to hold the current batch of translated rows

        # Process rows within the batch
        for j in range(i, min(i + batch_size, len(data_ru))):
            row = data_ru.iloc[j]

            # Translate dialog and summary
            dialog_translated = split_and_translate(row['dialog'])
            summary_translated = split_and_translate(row['summary'])

            # Append the translated data to the batch
            batch_data.append({
                'dialog_translated': dialog_translated,
                'summary_translated': summary_translated
            })

            # Log progress
            print(f"Row {j + 1}/{len(data_ru)} completed.")
            print(f"Translated Dialog: {dialog_translated[:10]}...")
            print(f"Translated Summary: {summary_translated[:10]}...\n")

        # Convert batch data to DataFrame
        batch_df = pd.DataFrame(batch_data)

        # Append the batch to the existing file
        batch_df.to_csv(output_path, mode='a', header=(i == 0), index=False, encoding='utf-8')
        print(f"Batch {i // batch_size + 1} with {len(batch_data)} rows saved to file.")

    except Exception as e:
        print(f"Error in batch starting at row {i}: {e}")
        continue


Existing data with 220 rows found. Continuing processing.
Row 221/12460 completed.
Translated Dialog: What are y...
Translated Summary: Schureen <...

Row 222/12460 completed.
Translated Dialog: I'm Mickey...
Translated Summary: <speaker_1...

Row 223/12460 completed.
Translated Dialog: You've bee...
Translated Summary: <speaker_2...



KeyboardInterrupt: 

In [ ]:
# Check
data_en = '/content/drive/MyDrive/translated_data.csv'
data_en = pd.read_csv(data_en, encoding='utf-8')
data_en

,dialog_translated,summary_translated
0,"Hello, Mr. Smith. I'm Dr. Hawkins. Why are you...","Mr. Smith is undergoing an examination, and Dr..."
1,"Hello, Mrs. Parker, how are you? Hello, Dr. Pe...","Mrs. Parker takes Ricky on the shots, Dr. Pete..."
2,"I'm sorry, you haven't seen a set of keys? Oka...",<speaker_1> looks for a set of keys and asks <...
3,Why didn't you tell me you had a girlfriend? Y...,#speaker_1> is angry because <speaker_2> did n...
4,"Watsap, ladies, tonight you will look beautifu...","Malik invites Nicky to the dance, Nicky agrees..."
...,...,...
12455,"You're Mr Green of Manchester, aren't you?","Tan Ling picks up Mr Greene, who's easily reco..."
12456,Mr. Ewing said we should show up at the confer...,We're planning to take the subway to the confe...
12457,"Let's see what we can find. We have a big car,...",<speaker_2> rents a small car for five days us...
12458,"Well, my mom lost her job yesterday. Yeah, it'...",Mother <speaker_2> lost her job. <speaker_2> h...


In [ ]:
# Initialize the Sentence-BERT model
model = SentenceTransformer('efederici/sentence-bert-base')

# Function to vectorize text
def vectorize_text(text):
    return model.encode(text, convert_to_tensor=False)

# Vectorize dialog and summary columns
data_en['dialog_vector'] = data_en['dialog_translated'].apply(vectorize_text)
data_en['summary_vector'] = data_en['summary_translated'].apply(vectorize_text)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.43k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/725k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
data_en

,dialog_translated,summary_translated,dialog_vector,summary_vector,similarity,similarity_analysis
0,"Hello, Mr. Smith. I'm Dr. Hawkins. Why are you...","Mr. Smith is undergoing an examination, and Dr...","-0.20648655,-0.39049715,0.114257775,0.9659268,...","-0.2124087,-0.4145447,-0.017479986,0.23434746,...",0.652867,OK
1,"Hello, Mrs. Parker, how are you? Hello, Dr. Pe...","Mrs. Parker takes Ricky on the shots, Dr. Pete...","-0.4501079,-0.29434282,-0.36100984,0.9804756,0...","-0.8513125,-0.12612203,-0.3537028,0.26882616,0...",0.670589,OK
2,"I'm sorry, you haven't seen a set of keys? Oka...",<speaker_1> looks for a set of keys and asks <...,"-0.42546865,-0.34919757,-0.26658517,0.59293866...","-0.34504086,-0.4638284,-0.14432041,0.54139316,...",0.628792,OK
3,Why didn't you tell me you had a girlfriend? Y...,#speaker_1> is angry because <speaker_2> did n...,"-0.46212667,-0.3158803,0.049964312,0.7160261,0...","-0.31433362,-0.47736472,0.24798639,0.73971623,...",0.494523,Mismatch
4,"Watsap, ladies, tonight you will look beautifu...","Malik invites Nicky to the dance, Nicky agrees...","-0.443559,-0.89411443,0.13211513,0.83620864,0....","-0.62073094,-0.5877532,0.13404904,0.60098785,-...",0.722509,OK
...,...,...,...,...,...,...
12455,"You're Mr Green of Manchester, aren't you?","Tan Ling picks up Mr Greene, who's easily reco...","-0.5247632,0.1352413,0.0026963,1.1287378,-0.20...","-0.82646286,-0.5858165,-0.3999558,0.85818565,0...",0.506816,OK
12456,Mr. Ewing said we should show up at the confer...,We're planning to take the subway to the confe...,"-0.5702838,-0.06678255,0.13421737,0.43816996,0...","-0.66142184,-0.26701692,-0.4373265,0.55869144,...",0.609372,OK
12457,"Let's see what we can find. We have a big car,...",<speaker_2> rents a small car for five days us...,"-0.3295982,0.032088816,-0.2866894,0.6896831,0....","-0.4964086,-0.48960876,-0.224866,0.509808,0.26...",0.578926,OK
12458,"Well, my mom lost her job yesterday. Yeah, it'...",Mother <speaker_2> lost her job. <speaker_2> h...,"-0.3485174,-0.63580704,-0.22024956,0.92049,-0....","-0.45422062,-0.38650203,0.16428088,0.76885575,...",0.503839,OK


In [ ]:
def compute_similarity(row):
    # Convert the string representations of vectors back to lists of floats
    dialog_vector = np.array(list(map(float, row['dialog_vector'].split(','))))
    summary_vector = np.array(list(map(float, row['summary_vector'].split(','))))

    # Compute cosine similarity
    return cosine_similarity([dialog_vector], [summary_vector])[0][0]

# Calculate cosine similarity
data_en['similarity'] = data_en.apply(compute_similarity, axis=1)

# Mark mismatches based on similarity threshold
threshold = 0.5  # Adjust threshold as needed
data_en['similarity_analysis'] = data_en['similarity'].apply(lambda x: 'Mismatch' if x < threshold else 'OK')

# Filter out mismatches
data_en = data_en[data_en['similarity_analysis'] != 'Mismatch']

# Convert vectors into numpy arrays
dialog_vectors_np = np.array(data_en['dialog_vector'].apply(lambda x: np.fromstring(x, sep=',')).tolist())
summary_vectors_np = np.array(data_en['summary_vector'].apply(lambda x: np.fromstring(x, sep=',')).tolist())

# Create sparse matrices for dialog and summary vectors
dialog_sparse_matrix = sp.csr_matrix(dialog_vectors_np)
summary_sparse_matrix = sp.csr_matrix(summary_vectors_np)

# Combine the dialog and summary vectors into a single sparse matrix
combined_sparse_matrix = sp.hstack([dialog_sparse_matrix, summary_sparse_matrix])

npz_file_path = '/content/drive/MyDrive/summary_vector_db.npz'
sp.save_npz(npz_file_path, combined_sparse_matrix)

print(f"Processing complete! Combined vector DB saved to {npz_file_path}")


Processing complete! Combined vector DB saved to /content/drive/MyDrive/summary_vector_db.npz


In [ ]:
# Check
data_ve = '/content/drive/MyDrive/summary_vector_data.csv'
data_ve = pd.read_csv(data_ve, encoding='utf-8')
data_ve

  (0, 0)	-0.20648655
  (0, 1)	-0.39049715
  (0, 2)	0.114257775
  (0, 3)	0.9659268
  (0, 4)	0.71527
  (0, 5)	-0.05188588
  (0, 6)	-0.72237325
  (0, 7)	0.082433924
  (0, 8)	0.15992063
  (0, 9)	-0.31606814
  (0, 10)	-0.10605106
  (0, 11)	0.14192985
  (0, 12)	-0.27953273
  (0, 13)	0.58792746
  (0, 14)	-0.02914636
  (0, 15)	0.7587449
  (0, 16)	-0.48395494
  (0, 17)	0.16664228
  (0, 18)	0.087844186
  (0, 19)	0.51152813
  (0, 20)	-0.1418636
  (0, 21)	0.39994702
  (0, 22)	0.31776938
  (0, 23)	-0.55501246
  (0, 24)	0.2517217
  :	:
  (9690, 1511)	0.19846343
  (9690, 1512)	-0.28763407
  (9690, 1513)	-0.06871212
  (9690, 1514)	-0.056144275
  (9690, 1515)	-0.1301485
  (9690, 1516)	0.07871826
  (9690, 1517)	0.43992805
  (9690, 1518)	0.26923308
  (9690, 1519)	-0.24312201
  (9690, 1520)	-0.15549618
  (9690, 1521)	0.21071288
  (9690, 1522)	-0.1009526
  (9690, 1523)	0.060095627
  (9690, 1524)	-0.12441099
  (9690, 1525)	-0.026039306
  (9690, 1526)	-0.03963616
  (9690, 1527)	0.35831088
  (9690, 1528)	0.14